In [ ]:
from matplotlib import pyplot as plt
import numpy as np
import pandas as pd

import pandas as pd
import numpy as np

from constants import ANALYSIS_ROOT

In [ ]:
df = pd.read_csv(ANALYSIS_ROOT / 'Hao_PBMC/cell_typing_dilution.tsv', delimiter='\t')

In [ ]:
df['representation_type'].value_counts()

In [ ]:
# Full-depth references follow the current main benchmark results.
full_depth_f1 = (
    pd.read_csv(ANALYSIS_ROOT / "Hao_PBMC/cell_typing.tsv", sep="\t")
    .groupby("representation_type")["macro_f1"].mean()
)
BEST_BASELINE = full_depth_f1.loc[[
    "scvi_emb_10", "scvi_emb_50", "scvi_emb_512",
    "harmony_pca_10", "harmony_pca_50", "harmony_pca_512",
    "magic_50_markers", "scvi_50_markers",
]].max()
BEST_FOUNDATIONAL_MODEL = full_depth_f1.loc[[
    "scfoundation", "scgpt", "scimilarity", "uce_4l_1280", "uce_33l_1280",
]].max()
FULL_RAW_RNA = full_depth_f1["rna_log1p"]
BEST_OUR = full_depth_f1["scunveil_emb_2048"]


In [ ]:
ANN_VOCAB = {
    'scunveil_emb_2048': 'scUNVEIL',
    'rna_log1p': 'Raw RNA',
    
}

In [ ]:
# Make sure dilution and macro_f1 are numeric
df["dilution"] = pd.to_numeric(df["dilution"])
df["macro_f1"] = pd.to_numeric(df["macro_f1"])

# Compute mean and std for each representation type and dilution
summary = (
    df.groupby(["representation_type", "dilution"], as_index=False)
      .agg(
          macro_f1_mean=("macro_f1", "mean"),
          macro_f1_std=("macro_f1", "std"),
          n=("macro_f1", "size")
      )
)

# If a group has only one replicate, std will be NaN
summary["macro_f1_std"] = summary["macro_f1_std"].fillna(0)

x_ours = None
y_ours = None

# Plot
plt.figure(figsize=(6, 4))
x_min = np.inf
x_max = -np.inf
for representation_type, sub in sorted(summary.groupby("representation_type"), key=lambda x: x[0])[::-1]:

    sub = sub.sort_values("dilution")

    x = sub["dilution"].to_numpy()
    y = sub["macro_f1_mean"].to_numpy()
    yerr = sub["macro_f1_std"].to_numpy()

    print(f'x_{representation_type}', x)
    print(f'y_{representation_type}', y)

    if representation_type == 'scunveil_emb_2048':
        x_ours = x.copy()
        y_ours = y.copy()

    plt.plot(
        x,
        y,
        marker="o",
        linewidth=2,
        label=ANN_VOCAB[representation_type]
    )

    plt.fill_between(
        x,
        y - yerr,
        y + yerr,
        alpha=0.2
    )

    x_max = np.max([x_max, x.max()])
    x_min = np.min([x_min, x.min()])


# plt.hlines(BEST_OUR, x_min, x_max, colors='r', linestyles='--', label='Ours')
plt.hlines(BEST_BASELINE, x_min, x_max, colors='k', linestyles='--', label='Best Baseline')
plt.hlines(BEST_FOUNDATIONAL_MODEL, x_min, x_max, colors='k', linestyles=':', label='Best Foundational Model')
plt.hlines(FULL_RAW_RNA, x_min, x_max, colors='tab:orange', linestyles=':', label='Best Raw RNA')

plt.yticks(np.linspace(0, 1, 11))
plt.xscale("log")
plt.xlabel("Retention Probability")
plt.ylabel("Macro F1")
# plt.title("Macro F1 vs Retention Probability")
plt.legend(title=r"$\mathbf{Representation\ type}$", framealpha=1.0)
plt.grid(True, which="both", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.ylim(0, 0.85)

plt.savefig(ANALYSIS_ROOT / "Hao_PBMC/figures/macro_f1_vs_retention.svg", dpi=300)
plt.savefig(ANALYSIS_ROOT / "Hao_PBMC/figures/macro_f1_vs_retention.pdf")
plt.show()

In [ ]:
def logx_crossing(reference_constant, curve_x, curve_y):
    x = np.asarray(curve_x)
    y = np.asarray(curve_y)

    d = y - reference_constant
    idx = np.where(d[:-1] * d[1:] <= 0)[0]

    if len(idx) == 0:
        return None

    i = idx[0]

    if d[i] == 0:
        return x[i]

    logx = np.log10(x)

    t = -d[i] / (d[i + 1] - d[i])
    logx_cross = logx[i] + t * (logx[i + 1] - logx[i])

    return 10 ** logx_cross

In [ ]:
def print_umi_efficiency(reference_name, crossing_ratio):
    umi_reduction_pct = 100 * (1 - crossing_ratio)

    print(
        f"scUNVEIL achieves the same results with "
        f"{umi_reduction_pct:.0f}% fewer UMIs than {reference_name}."
    )


print_umi_efficiency(
    "the best foundational model",
    logx_crossing(BEST_FOUNDATIONAL_MODEL, x_ours, y_ours),
)

print_umi_efficiency(
    "raw measurements",
    logx_crossing(FULL_RAW_RNA, x_ours, y_ours),
)